# QABB — OCR the scanned / no-text documents

These 14 files in your QABB archive are **scanned images with no searchable text layer**, so full-text search can't reach inside them. This notebook runs entirely in **your own Google Drive** (where there are no file-size limits), adds a searchable text layer to each, and writes the results to a new **`QABB OCR output`** folder — nothing is overwritten.

**How to use:** Runtime ▸ Run all, approve the Drive permission popup, wait (~10–20 min; the two 106 MB files dominate). At the end you'll get a `QABB_ocr_results.csv` mapping — **send that back to Claude** to fold the OCR'd copies into the searchable index.

> Tip: Runtime ▸ Change runtime type ▸ **CPU** is fine (no GPU needed).


In [ ]:
# 1) Install OCR toolchain (~1-2 min)
!sudo apt-get -qq update
!sudo apt-get -qq install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-fra \
     ghostscript qpdf unpaper pngquant libreoffice-writer >/dev/null
!pip -q install ocrmypdf
import ocrmypdf, shutil
print("ocrmypdf", ocrmypdf.__version__, "| tesseract langs:", end=" ")
!tesseract --list-langs 2>/dev/null | tail -n +2 | tr '\n' ' '


In [ ]:
# 2) Authenticate to YOUR Google Drive
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import io, os, time
drive = build('drive', 'v3')
print("Authenticated as the signed-in Google account.")


In [ ]:
# 3) The files to OCR (the 14 flagged 'scanned / no-text' documents)
PDF_TARGETS = [
  {
    "id": "1vvFCs2JBJ5GcTTyOsiaj2n78TyOo1vrn",
    "name": "Meek_1913_Naturalist.pdf"
  },
  {
    "id": "1FU420HM8Cb1uS8mRRE91dC793BdS1i4d",
    "name": "1991_Prior_LET.pdf"
  },
  {
    "id": "1dRdXhmOjrU6QEpQ6kLXgJc2ah4mCoD74",
    "name": "Malaisa_2009_CapeWardHunt.pdf"
  },
  {
    "id": "1YGnNhLOw7UwFQTVZ-WuL-zDzrsSapZSs",
    "name": "1989_IFTA_LET.pdf"
  },
  {
    "id": "15Qd7zprl4cI09lPVcNuLx11XpjdwoOG8",
    "name": "Anon_1984_Kumusi.pdf"
  },
  {
    "id": "1Lj4Sgd01Uk83E99FnmlkYTmOmEE5ZEtt",
    "name": "1987_WWF_LET.pdf"
  },
  {
    "id": "1ZAY34UYh4ZdRbAkC-LxRgz2KK59pS61c",
    "name": "1989_WWF_LET.pdf"
  },
  {
    "id": "1sYCOXwuD05ePvXhmpT8smfiNrBYmNdjN",
    "name": "1989_DEC_LET.pdf"
  },
  {
    "id": "11HL9iMPCJx6JUdNzKqRaVdKaqUb3oElF",
    "name": "1991_Parsons_LET1.pdf"
  },
  {
    "id": "1T-Q4OG4zXZ6p_rEZaIVsktZhswlbekEO",
    "name": "1988_Parsons_LET.pdf"
  },
  {
    "id": "1pt80zo2hdqLPrG2K3OLAxfw8zNIxWZxV",
    "name": "Blandin_1973_Fr.pdf"
  },
  {
    "id": "1R6woSmAluvlkeNxJN-6QnsExAmZSZ0FB",
    "name": "1989_Wilmot_MIN.pdf"
  },
  {
    "id": "1E2o3zdNsffztRkG4yOe1GfDTdwAiDDPB",
    "name": "1991_Atkins_LET.pdf"
  }
]

DOCX_TARGETS = [
  {
    "id": "1l6M26vMyw0wSJrgLTwdU6JMKN_lt7WmT",
    "name": "O_alexandrae_MANUSCRIPT.docx"
  }
]

print(len(PDF_TARGETS), 'scanned PDFs +', len(DOCX_TARGETS), 'Office doc(s) to process')

In [ ]:
# 4) Create the output folder in your Drive: "QABB OCR output"
def get_or_create_folder(name, parent=None):
    q = f"name='{name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
    if parent: q += f" and '{parent}' in parents"
    r = drive.files().list(q=q, fields="files(id,name)").execute().get("files", [])
    if r: return r[0]["id"]
    meta = {"name": name, "mimeType": "application/vnd.google-apps.folder"}
    if parent: meta["parents"] = [parent]
    return drive.files().create(body=meta, fields="id").execute()["id"]

OUT_FOLDER_ID = get_or_create_folder("QABB OCR output")
print("Output folder ready:  https://drive.google.com/drive/folders/" + OUT_FOLDER_ID)


In [ ]:
# 5) Helpers: download by id, OCR, upload back
os.makedirs("in", exist_ok=True); os.makedirs("out", exist_ok=True)

def download(file_id, dest):
    req = drive.files().get_media(fileId=file_id)
    with io.FileIO(dest, "wb") as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=16*1024*1024)
        done = False
        while not done:
            _, done = dl.next_chunk()

def upload_pdf(local, name, parent):
    meta = {"name": name, "parents": [parent]}
    media = MediaFileUpload(local, mimetype="application/pdf", resumable=True)
    return drive.files().create(body=meta, media_body=media, fields="id,webViewLink").execute()

def ocr_pdf(src_pdf, out_pdf):
    # scans have no text layer -> force-ocr guarantees one; rotate+deskew fixes orientation
    ocrmypdf.ocr(src_pdf, out_pdf, language="eng+fra",
                 rotate_pages=True, deskew=True, force_ocr=True,
                 progress_bar=False, jobs=2)


In [ ]:
# 6) Run OCR on every target.  Output name gets an "_OCR" suffix so nothing is overwritten.
import subprocess, csv
results = []

def process(file_id, name, is_docx=False):
    stem = os.path.splitext(name)[0]
    raw = f"in/{stem}" + (".docx" if is_docx else ".pdf")
    src_pdf = f"in/{stem}.pdf"
    out_pdf = f"out/{stem}_OCR.pdf"
    print(f"\n▶ {name}")
    download(file_id, raw)
    if is_docx:
        # convert Word -> PDF first, then OCR (manuscript has no extractable text layer)
        subprocess.run(["libreoffice","--headless","--convert-to","pdf","--outdir","in", raw],
                       check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    try:
        ocr_pdf(src_pdf, out_pdf)
    except Exception as e:
        print("   force-ocr failed, retrying with redo-ocr:", e)
        ocrmypdf.ocr(src_pdf, out_pdf, language="eng+fra", rotate_pages=True,
                     deskew=True, redo_ocr=True, progress_bar=False, jobs=2)
    up = upload_pdf(out_pdf, f"{stem}_OCR.pdf", OUT_FOLDER_ID)
    mb = os.path.getsize(out_pdf)/1e6
    print(f"   ✓ OCR done ({mb:.1f} MB) → {up['webViewLink']}")
    results.append({"original_name": name, "source_id": file_id,
                    "ocr_name": f"{stem}_OCR.pdf", "ocr_id": up["id"],
                    "ocr_url": up["webViewLink"]})

for t in PDF_TARGETS:
    try: process(t["id"], t["name"])
    except Exception as e: print("   ✗ FAILED:", e); results.append({"original_name":t["name"],"source_id":t["id"],"ocr_id":"","ocr_url":f"ERROR: {e}"})

for t in DOCX_TARGETS:
    try: process(t["id"], t["name"], is_docx=True)
    except Exception as e: print("   ✗ FAILED:", e); results.append({"original_name":t["name"],"source_id":t["id"],"ocr_id":"","ocr_url":f"ERROR: {e}"})

print(f"\nDone: {sum(1 for r in results if r.get('ocr_id'))}/{len(results)} OCR'd.")


In [ ]:
# 7) Save the mapping (old id -> OCR'd id) and upload it next to the OCR'd files.
#    >>> Send the printed table (or the CSV) back to Claude so it can fold these into the index. <<<
import csv
with open("QABB_ocr_results.csv","w",newline="") as fh:
    w=csv.DictWriter(fh, fieldnames=["original_name","source_id","ocr_name","ocr_id","ocr_url"])
    w.writeheader(); w.writerows(results)
upload = drive.files().create(
    body={"name":"QABB_ocr_results.csv","parents":[OUT_FOLDER_ID]},
    media_body=MediaFileUpload("QABB_ocr_results.csv", mimetype="text/csv"),
    fields="id,webViewLink").execute()
print("Mapping CSV saved to Drive:", upload["webViewLink"])
print()
for r in results:
    print(f"{r['original_name']:38}  ->  {r.get('ocr_id','')}")
from google.colab import files as _f
_f.download("QABB_ocr_results.csv")
